In [123]:
import pandas as pd
import random
import numpy as np
from tmdbv3api import Movie, Person, TMDb
import requests

filmography_cache = {}

In [124]:
def big5_Oscar_df(path, begin_year = 1960, end_year = 2024):
    '''
    Function reads in a csv containing data on oscar nominees and converts into a pandas dataframe
    
    Param: path of csv file
    Returns: pandas dataframe
    
    '''

    #Take in oscar nominee csv
    oscar_df = pd.read_csv(path)

    #select for all columns that contain strings
    str_cols = oscar_df.select_dtypes('str').columns
    #convert all rows of each column with str type to lowercase
    oscar_df[str_cols] = oscar_df[str_cols].apply(lambda x: x.str.lower())
    # handle cases of multiple nominees, giving each their own row
    oscar_df['name'] = oscar_df['name'].astype(str)
    oscar_df['name'] = oscar_df['name'].str.split('/')
    oscar_df = oscar_df.explode('name')
    # pre-process role column to be usable later
    role_dict = {
        'actor':'actor', 
        'actress': 'actress',
        'director': 'director',
        'writer':'writer',
    }
    
    for key, value in role_dict.items():
        oscar_df.loc[oscar_df['role'].str.contains(key), 'role'] = value

    # limit df to only pull from big five categories
    oscar_df = oscar_df[oscar_df['role'].isin(role_dict.keys()) | (oscar_df['category'] == 'best picture')]

    

    oscar_df = oscar_df[oscar_df['year_ceremony'].between(begin_year, end_year)]
        
    

    return oscar_df

path_to_data = '../data/the_oscar_award.csv'

oscar_df = big5_Oscar_df(path_to_data, 2024, 2024)

In [125]:
class category: 

    def __init__(self, df, category):
        
        self.category = category
        self.df = df[df['category'].str.contains(self.category)]
        self.movie_list = list(self.df['film'])
        


In [126]:
class nominee: 

    def __init__(self, name, df, cat, role, cat_to_role_dict, filmography_cache):

        self.name = name
        self.role = role
        self.cat_to_role_dict = cat_to_role_dict
        self.o_df = df

        self.c = category(df, cat)
        # Isolate only the oscar movies the nominee has appeared in for this category (even if not the awards intended recipient)

        


    def _get_filmography(self):

        
        if self.name in filmography_cache:
            return filmography_cache[self.name]
        
        api_key = 'f49d2ebf0d11031312ade120f61c513d'

        # Search for person directly via API
        search_url = "https://api.themoviedb.org/3/search/person"
        search_params = {"api_key": api_key, "query": self.name}
        search_data = requests.get(search_url, params=search_params).json()

        results = search_data.get("results", [])
        
        if not results:  # guard against no match
            return []

        nomID = results[0]["id"]  # ✅ plain dict, no .id attribute needed

        # Get credits
        credits_url = f"https://api.themoviedb.org/3/person/{nomID}/combined_credits"
        credits_params = {"api_key": api_key}
        data = requests.get(credits_url, params=credits_params).json()

        movies = data.get("cast", [])
        filmography = [m["title"].lower() for m in movies if "title" in m]
            

        filmography_cache[self.name] = filmography

        return filmography
    
    def set_self_df(self):

        if self.cat_to_role_dict[self.c.category] == self.role:
            self.df = self.o_df[(self.o_df['role'] == self.role) & (self.o_df['name'] == self.name)]
        else: 
            self.df = self.o_df[self.o_df['film'].isin(self._get_filmography())] 

        return self.df   
        

    def nom_cat_appearances(self):

        self.df = self.set_self_df()

        self.nom_category_appearance = self.df[
        self.df['film'].isin(self.c.df['film']) & 
        self.df['category'].isin(self.c.df['category'])
        ]    

        return self.nom_category_appearance
    

    def oscars_score(self):

        '''
        function calculates a weight based on oscar nominations and wins. Wins are heavily favored, but nominations are not counted against as they would be in a probabilistic determination 
        of success 
        
        param: self <- object of nominee class
        returns: weighted oscar score
        
        '''


        self.nom_category_appearance = self.nom_cat_appearances()
        
        # Isolate only the winners from the category nominations the nominee has appeared in 
        outcomes = list(self.nom_category_appearance['winner'])

        # initialize oscar score
        oscar_score = 0

        # If nominee has no previous nominations for category
        if len(outcomes) == 0:
            return 0
        else:
            # Iterate through each movie's outcome and add weighted score to oscar score
            for decision in outcomes:
                if self.cat_to_role_dict[self.c.category] == self.role:
                    if decision == True: 
                    #arbitrary weighting, favorable weighting for a win
                        oscar_score += 10
                    else: 
                        oscar_score += 5
                elif self.cat_to_role_dict[self.c.category] != self.role:
                    if decision == True:
                        oscar_score += 3
                    else: 
                        oscar_score += 1
                else: 
                    #arbitrary weighting, less favorable but still positive weighting for a nomination
                    oscar_score += 0

        return oscar_score

    def synergy_boost(self, crew_list):

        '''
        Function considers past professional relationships with other nominees and calculates a synergy boost.

        param1: self <- object of nominee class
        param2: crew_list <- list of other crew members on movie, each item in list is an instance of the nominee class

        returns: new score with added synergy boost

        '''
        self.films = set(self._get_filmography())
        
        self.df = self.set_self_df(

        )
        # Initialize empy list for common movies
        all_common_movies = []

        # Initialize synergy score
        synergy_score = 0 

        # Iterate through crew list
        for member in crew_list:
            # Exclude self
            if member.name != self.name:
                # Takes the intersection between self and other crew members filmoraphy
                common_movies = set(member._get_filmography()) & set(self.films)
                # if 1 or more movies in common
                if len(common_movies) > 0:
                    # iterate through common movies
                    for movies in common_movies: 
                        # add to list of common movies shared between self and all other nominees 
                        all_common_movies.append(movies)        
                else:
                    # If no movies in common return a synergy score of 0
                    continue
            else: 
                continue
                
        for movie in all_common_movies:
            row = self.df[self.df["film"] == movie]

            if row.empty:
                continue  # movie wasn’t nominated at all

            if row["winner"].iloc[0] is True:
                synergy_score += 0.6
            else:
                synergy_score += 0.2

        return synergy_score
    
    def __repr__(self):
        
        return self.name

In [127]:
class rand_movie: 

    def __init__(self, category, talent_pool, oscar_df, cat_to_role_dict):
            self.category = category
            self.oscar_df = oscar_df
            self.talent_pool = talent_pool
            self.cat_to_role_dict = cat_to_role_dict 

            
            self.nominee_cat = cat_to_role_dict[category]
            



    def hire_crew(self, role):

        pool = list(self.talent_pool[role])

        rng = np.random.default_rng()

        i = rng.integers(low = 0,  high = len(pool))

        name = pool[i]

        return nominee(name, self.oscar_df, self.category, role, self.cat_to_role_dict, filmography_cache)
    


    def credits(self):

        

        self.actor = self.hire_crew('actor')

        self.actress = self.hire_crew('actress')

        self.director = self.hire_crew('director')

        self.writer = self.hire_crew('writer')

        self.crew_dict = {
            'actor': self.actor, 
            'actress': self.actress,
            'director': self.director,
            'writer': self.writer, 
        }


        self.nominee = self.crew_dict[self.nominee_cat]
        return self.crew_dict, 
    
    def movie_oscar_score(self): 
        
        crew_list = list(self.crew_dict.values())
        score = 0 

        for crew in self.crew_dict.values():  
            oscar_score = crew.oscars_score()
            synergy_score = crew.synergy_boost(crew_list)
            score += oscar_score + synergy_score
    
        return score  
    
    def __repr__(self): 

        return str(list(self.crew_dict.values()))

In [128]:
def metropolis_hastings(category, talent_pool, oscar_df, cat_to_role_dict, iterations = 1000):
    c = 0 
    i = 1
    init_movie = rand_movie(category, talent_pool, oscar_df, cat_to_role_dict)
    init_crew = init_movie.credits()
    init_score = init_movie.movie_oscar_score()
    conv_check = np.inf

    while i <= iterations:


        init_score = init_movie.movie_oscar_score()

        

        nom_movie = rand_movie(category, talent_pool, oscar_df, cat_to_role_dict)
        nom_crew = nom_movie.credits()
        nom_score = nom_movie.movie_oscar_score()

        a = min(1, nom_score/init_score)

        if random.random() < a: 
            init_movie = nom_movie
            init_score = nom_score

        

        # print(init_movie)
        
        i += 1
    
    return init_movie


In [129]:
def sim_oscar_season(category_list, oscar_df, talent_pool, cat_to_role_dict):

    oscar_season = {}
    nom_movies = 5
    for cat in category_list:
        nominees = []
        for n in range(nom_movies):
            nominee = metropolis_hastings(cat, talent_pool, oscar_df, cat_to_role_dict)
            nominees.append(nominee)
        oscar_season[cat] = nominees

    return oscar_season



In [130]:
def academy_award_show(year_start = 1960, year_end = 2024): 

    print('welcome to the forever oscars, the nominees for each category have been selected to represent the best of the best!')
    print('they are each supported by an all star cast that have a history of support in the given category')
    print('who will reign supreme!')

    path_to_data = '../data/the_oscar_award.csv'

    oscar_df = big5_Oscar_df(path_to_data, year_start, year_end)

    cat_to_role_dict = {
            'actor':'actor', 
            'actress': 'actress',
            'directing': 'director',
            'writing':'writer',
        }
    
    init_talent_pool = (oscar_df.groupby('role')['name'].apply(list).to_dict())

    talent_pool = {}
    for k, v in init_talent_pool.items(): 
        talent_pool[k] = list(set(v))

    category_list = [x for x in cat_to_role_dict.keys() if oscar_df['category'].str.contains(x).any()]
    
    oscar_season = sim_oscar_season(category_list, oscar_df, talent_pool, cat_to_role_dict)

    for cat in oscar_season.keys():
        weights = []
        for idx, movie in enumerate(oscar_season[cat]):
            print(f'Nominated for the category of {cat} is {movie.nominee}')
            weight = movie.movie_oscar_score()
            weights.append(weight)
        winner = random.choices(oscar_season[cat],weights)[0]
        print(f'and the winner is.... {winner.nominee}')

    return oscar_season


oscar_season = academy_award_show(1960,2024)



# #for cat in oscar_season.keys():
#     for nominee in oscar_season[cat]:
#         print(f'Nominated for the category of {cat} is {oscar_season[cat].cat_to_role_dict[cat].actor}')


welcome to the forever oscars, the nominees for each category have been selected to represent the best of the best!
they are each supported by an all star cast that have a history of support in the given category
who will reign supreme!
Nominated for the category of actor is charlton heston
Nominated for the category of actor is anthony hopkins
Nominated for the category of actor is clint eastwood
Nominated for the category of actor is ed harris
Nominated for the category of actor is melvyn douglas
and the winner is.... ed harris
Nominated for the category of actress is june squibb
Nominated for the category of actress is helen mirren
Nominated for the category of actress is toni collette
Nominated for the category of actress is piper laurie
Nominated for the category of actress is sally hawkins
and the winner is.... june squibb
Nominated for the category of directing is pawel pawlikowski
Nominated for the category of directing is william wyler
Nominated for the category of directing i